<a href="https://colab.research.google.com/github/x1001000/Colab-Notebooks/blob/main/mp3_to_srt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NotebookLM wav to mp3/mp4
- https://cloudconvert.com/wav-to-mp3
- https://www.onlineconverter.com/audio-to-video

# mp3 to srt

In [ ]:
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
audio_file= open("podcast.mp3", "rb")

transcription = client.audio.transcriptions.create(
    model="whisper-1",
    response_format="verbose_json",
    timestamp_granularities=["word"],
    file=audio_file
)

In [ ]:
lines = transcription.text.split()
timestamps = transcription.model_dump()['words']
the_end = timestamps[-1]['end']

In [ ]:
starts = []
scan_from = 0
for line in lines:
    for timestamp in timestamps[scan_from:]:
        scan_from += 1
        if line.startswith(timestamp['word']):
            start = timestamp['start']
            starts.append(start)
            break

In [ ]:
def srt_timestamp_format(t):
    hours, remainder = divmod(t, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{int(hours):02}:{int(minutes):02}:{seconds:06.3f}"

In [ ]:
srt_lines = ''
for i, line in enumerate(lines):
    start = srt_timestamp_format(starts[i])
    end = srt_timestamp_format(starts[i+1] if i+1 < len(starts) else the_end)
    srt_lines += f"{i+1}\n{start} --> {end}\n{line}\n\n"

In [ ]:
with open("podcast.srt", "w") as f:
    f.write(srt_lines)

In [ ]:
from google.colab import files
files.download('podcast.srt')

# srt to mp4
- https://www.happyscribe.com/subtitle-tools/add-srt-to-mp4